# 02 - 模型實驗 (Model Experiments)

富邦金控 MA 甄選 — 題目二：客戶意圖分類

實驗流程：
1. TF-IDF + Logistic Regression
2. TF-IDF + LinearSVC
3. 超參數調整 (GridSearchCV)
4. 交叉驗證
5. 錯誤分析
6. 多意圖拆解展示

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.data_preprocess import TextPreprocessor, TfidfFeatureBuilder, load_dataset
from src.model_trainer import TraditionalMLTrainer
from src.evaluator import IntentEvaluator
from app.llm_handler import MultiIntentHandler

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1. 資料準備

In [ ]:
raw_dir = '../data/raw'
train_file = [f for f in os.listdir(raw_dir) if 'train' in f][0]
eval_file = [f for f in os.listdir(raw_dir) if 'eval' in f][0]

train_texts, train_labels, _ = load_dataset(os.path.join(raw_dir, train_file))
eval_texts, eval_labels, _ = load_dataset(os.path.join(raw_dir, eval_file))

pp = TextPreprocessor()
train_clean = pp.transform_batch(train_texts)
eval_clean = pp.transform_batch(eval_texts)

fb = TfidfFeatureBuilder()
X_train, X_eval = fb.fit_transform(train_clean, eval_clean)
print(f"特徵維度: {fb.get_feature_dims()}")

## 2. Logistic Regression

In [ ]:
lr = TraditionalMLTrainer('logistic_regression', C=5.0)
result = lr.train(X_train, train_labels)
print(f"Train: {result}")

lr_preds = lr.predict(X_eval)
lr_eval = IntentEvaluator(eval_labels, lr_preds.tolist(), eval_texts)
print(f"Eval Accuracy: {lr_eval.overall_accuracy():.4f}")

## 3. LinearSVC

In [ ]:
svm = TraditionalMLTrainer('linear_svc', C=1.0)
result = svm.train(X_train, train_labels)
print(f"Train: {result}")

svm_preds = svm.predict(X_eval)
svm_eval = IntentEvaluator(eval_labels, svm_preds.tolist(), eval_texts)
print(f"Eval Accuracy: {svm_eval.overall_accuracy():.4f}")

## 4. 超參數調整 (GridSearchCV)

In [ ]:
gs = TraditionalMLTrainer('linear_svc')
gs_result = gs.grid_search(X_train, train_labels, {'C': [0.1, 0.5, 1.0, 2.0, 5.0]})
print(f"Best params: {gs_result['best_params']}")
print(f"Best CV score: {gs_result['best_score']:.4f}")

# Visualization
fig, ax = plt.subplots(figsize=(8, 4))
cs = [r['params']['C'] for r in gs_result['all_results']]
means = [r['mean_score'] for r in gs_result['all_results']]
stds = [r['std_score'] for r in gs_result['all_results']]
ax.errorbar(cs, means, yerr=stds, fmt='o-', capsize=5, linewidth=2)
ax.set_xlabel('C (Regularization)')
ax.set_ylabel('CV Accuracy')
ax.set_title('Hyperparameter Tuning: LinearSVC')
ax.set_xscale('log')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 最佳模型評估

In [ ]:
best_preds = gs.predict(X_eval)
evaluator = IntentEvaluator(eval_labels, best_preds.tolist(), eval_texts)
evaluator.print_summary()

In [ ]:
# 按領域準確率視覺化
domain_acc = evaluator.domain_accuracy()
fig, ax = plt.subplots(figsize=(10, 5))
domains = list(domain_acc.keys())
accs = [domain_acc[d]['accuracy'] for d in domains]
colors = plt.cm.Set2(np.linspace(0, 1, len(domains)))
ax.barh(domains, accs, color=colors, height=0.6)
for i, (d, a) in enumerate(zip(domains, accs)):
    ax.text(a + 0.005, i, f'{a:.2%}', va='center', fontweight='bold')
ax.set_xlabel('Accuracy')
ax.set_title('Accuracy by Domain (Best SVM)')
ax.set_xlim(0.75, 1.0)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. 交叉驗證

In [ ]:
cv_result = gs.cross_validate(X_train, train_labels, cv=5)
print(f"CV Scores: {[f'{s:.4f}' for s in cv_result['cv_scores']]}")
print(f"Mean: {cv_result['mean']:.4f} (±{cv_result['std']:.4f})")

## 7. 多意圖拆解展示

In [ ]:
# 建立分類函式
def classify_fn(text):
    clean = pp.transform(text)
    X = fb.transform([clean])
    intent = gs.predict(X)[0]
    decision = gs.model.decision_function(X)
    return intent, float(np.max(decision))

handler = MultiIntentHandler(classifier_fn=classify_fn)

test_queries = [
    "what's the weather like today",
    "check my balance and also book a flight to tokyo",
    "report my lost card and also freeze my account",
    "what's my credit limit and when is my bill due; also check the interest rate",
]

for q in test_queries:
    result = handler.analyze(q)
    print(f"\n  Input: \"{q}\"")
    print(f"  Compound: {result['is_compound']}")
    for sq in result['sub_queries']:
        print(f"    → \"{sq['text']}\" → {sq['intent']} (conf: {sq['confidence']:.2f})")

## 8. 儲存模型

In [ ]:
# 儲存最佳模型 & TF-IDF
gs.save('../models/svm_best.pkl')
fb.save('../models/tfidf_vectorizer.pkl')

# 儲存評估報告
evaluator.save_report('../models/eval_report.json')
print("所有模型與報告已儲存至 models/ 目錄")